In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import wandb

In [2]:
# Set publication style formatting
plt.style.use("seaborn-v0_8-paper" if "seaborn-v0_8-paper" in plt.style.available else "default")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

In [3]:
# 1. Setup output paths
try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent
except NameError:
    # Fallback for interactive notebook cells
    PROJECT_ROOT = Path.cwd()

# Ensure PROJECT_ROOT points to repo root if running inside the report/ folder
if PROJECT_ROOT.name == "report":
    PROJECT_ROOT = PROJECT_ROOT.parent

FIGURES_DIR = PROJECT_ROOT / "report" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


WANDB_PROJECT = "multimodal-brain-tumor-diagnostics"
api = wandb.Api()

print(f"Fetching run histories from W&B project: {WANDB_PROJECT}...")
runs = api.runs(WANDB_PROJECT)

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\aimen\_netrc.


Fetching run histories from W&B project: multimodal-brain-tumor-diagnostics...


In [4]:
# Collect run history dataframes
run_histories = {}
for run in runs:
    # Filter out empty or test runs
    if run.state in ["finished", "running" , "crashed"]:
        df_hist = run.history()
        if not df_hist.empty and "epoch" in df_hist.columns:
            run_histories[run.name] = {
                "history": df_hist,
                "config": run.config,
                "id": run.id
            }

In [5]:
fig, ax = plt.subplots(figsize=(7, 5))

for run_name, data in run_histories.items():
    df = data["history"]
    auc_col = "val/auc" if "val/auc" in df.columns else ("val_auc" if "val_auc" in df.columns else None)
    
    if auc_col and not df[auc_col].dropna().empty:
        ax.plot(df["epoch"], df[auc_col], marker='o', linewidth=2, label=run_name)

ax.set_title("Validation ROC-AUC Convergence Across Models")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation ROC-AUC")
ax.set_ylim([0.85, 1.0])
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(loc="lower right", frameon=True)

fig1_path = FIGURES_DIR / "auc_trajectories.png"
plt.savefig(fig1_path)
plt.close()
print(f"Saved: {fig1_path}")

Saved: c:\Users\aimen\OneDrive\Desktop\multimodal-brain-tumor-diagnostics\report\figures\auc_trajectories.png


In [6]:
fig, ax = plt.subplots(figsize=(8, 5))

for run_name, data in run_histories.items():
    df = data["history"]
    val_loss_col = "val/loss" if "val/loss" in df.columns else ("val_loss" if "val_loss" in df.columns else None)
    train_loss_col = "train/loss" if "train/loss" in df.columns else ("train_loss" if "train_loss" in df.columns else None)
    
    if val_loss_col and not df[val_loss_col].dropna().empty:
        line = ax.plot(df["epoch"], df[val_loss_col], label=f"{run_name} (Val)", linewidth=2)
        color = line[0].get_color()
        if train_loss_col and not df[train_loss_col].dropna().empty:
            ax.plot(df["epoch"], df[train_loss_col], linestyle="--", color=color, alpha=0.6, label=f"{run_name} (Train)")

ax.set_title("Train vs. Validation Loss Dynamics (Overfitting Benchmark)")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCE Loss")
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(loc="upper right", fontsize=8, frameon=True)

fig2_path = FIGURES_DIR / "loss_trajectories.png"
plt.savefig(fig2_path)
plt.close()
print(f"Saved: {fig2_path}")

Saved: c:\Users\aimen\OneDrive\Desktop\multimodal-brain-tumor-diagnostics\report\figures\loss_trajectories.png


In [7]:
summary_csv = PROJECT_ROOT / "report" / "tables" / "experiment_summary.csv"
if summary_csv.exists():
    df_summary = pd.read_csv(summary_csv)
    
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(
        data=df_summary, 
        x="Model Name", 
        y="Best Cls AUC", 
        palette="viridis", 
        ax=ax
    )
    
    ax.set_title("Peak Validation ROC-AUC Comparison")
    ax.set_ylabel("ROC-AUC")
    ax.set_ylim([0.90, 1.0])
    plt.xticks(rotation=15, ha="right")
    
    # Add numerical labels on top of bars
    for p in ax.patches:
        ax.annotate(
            f"{p.get_height():.4f}",
            (p.get_x() + p.get_width() / 2., p.get_height()),
            ha='center', va='bottom', fontsize=9, xytext=(0, 3),
            textcoords='offset points'
        )
        
    fig3_path = FIGURES_DIR / "metric_bar_chart.png"
    plt.savefig(fig3_path)
    plt.close()
    print(f"Saved: {fig3_path}")

print("\nAll report figures successfully generated in `report/figures/`")

C:\Users\aimen\AppData\Local\Temp\ipykernel_8764\3161642253.py:6: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(


Saved: c:\Users\aimen\OneDrive\Desktop\multimodal-brain-tumor-diagnostics\report\figures\metric_bar_chart.png

All report figures successfully generated in `report/figures/`
